<a href="https://colab.research.google.com/github/CIS5190/cis-4190-5190-fall-26/blob/main/hw4/hw4_skeleton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CIS 4190/5190 Fall 2026 - Homework 4**

# **Tips for Coding Homeworks**

1.   **Do not set/alter the random seeds we provide.** They exist so that everyone's numbers, and the hidden autograder tests, match up.
2.   Most bugs come down to minor errors in your code. Debugging ML code is a key skill you should acquire in this class: print shapes, and test each function on a tiny input before running it on the real data.
3.   **Every autograded function in this homework is graded on a small, fixed input**, computed by the grading cell right below it. The local test above each grading cell checks the same properties on the same kind of input, so if the local test passes and the grading cell does not, read the grader's message — it says which property failed.
4.   **A GPU is optional.** Section 1.1.7 trains for about 3 minutes on a CPU; nothing else trains for long. If you do switch runtime (*Runtime → Change runtime type*), re-run every cell from the top down to *Initialize the Autograder*.

Have fun!

> **Important: Save a Copy Before Starting**
> Please click *"Copy to Drive"* in the top menu bar or go to *File → Save a Copy in Drive*.
> This shared copy is read-only, so any edits you make to it will not be saved. Work in your
> own copy.

---

> **Points**
> This assignment is worth **50 points**: 42 points that every student completes, plus 8
> points marked **[5190 Extra]**, which are the four written questions in section 2.6.
>
> - **CIS 4190** is graded out of **42**. The [5190 Extra] parts are optional for you.
> - **CIS 5190** is graded out of **50**. The [5190 Extra] parts are required for you.
>
> CIS 4190 students are welcome to attempt the [5190 Extra] parts for credit, but your total
> for this assignment is still capped at 42 points.

---

> **How this notebook is graded**
> 38 of the 50 points are graded by the autograder in this notebook, from the
> PennGrader cells. The other 12 are graded by hand from the plots and written answers
> you leave in the notebook — sections 2.4 and 2.6.
> **Download the notebook with its outputs intact**, or there will be nothing there to mark.
>
> **Your most recent submission for each grading cell is the one that counts** — re-running a
> grading cell with a worse answer will lower your recorded score. You may resubmit each
> grading cell up to 100 times per day.
>
> **When you submit to Gradescope, it re-runs your notebook from the top, in order**, and every
> grading cell not wrapped in `if NOTEBOOK:` submits again. Since your latest submission counts,
> what Gradescope records for those cells is what your notebook produces when run top to bottom.
> An error stops the re-run at that cell.
>
> **Do not delete the `if NOTEBOOK:` lines.**

## Setup Cell 1: Imports and Configuration

In [ ]:
import os
import math
import random
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

np.random.seed(42)      # don't change this line!!
random.seed(42)         # don't change this line!!
torch.manual_seed(42)   # don't change this line!!

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

## Setup Cell 2: Environment Detection

- `NOTEBOOK` will be `True` if running in Colab and `False` in the autograder.
- Cells that start with `if NOTEBOOK:` are skipped by the autograder. Leave those guards in place.
- Do **not** modify this cell.

In [ ]:
# For autograder only, do not modify this cell.
NOTEBOOK = (os.getenv('IS_AUTOGRADER') is None)

if NOTEBOOK:
    print("[INFO, OK] Google Colab.")
else:
    print("[INFO, OK] Autograder.")

## PennGrader Setup

In [ ]:
%%capture
!pip3 install penngrader-client==0.5.13

In [ ]:
from penngrader.grader import *

### Autograder Setup

Please enter your **8-digit PennID** in the cell below:

In [ ]:
# PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY.
# IF NOT, THE AUTOGRADER WON'T KNOW WHO TO ASSIGN POINTS TO IN OUR BACKEND.

STUDENT_ID = 12345678          # YOUR PENN-ID GOES HERE AS AN INTEGER

## Initialize the Autograder

**NOTE:** We store your submissions and check them against other students' submissions. So, not that you would, but please do **not** cheat.

In [ ]:
%%writefile notebook-config.yaml

grader_api_url: "https://yj0vsrf8l4.execute-api.us-east-1.amazonaws.com/Initial/TestSandbox"
grader_api_key: "dBB5ucuV5L6GyKFKmlrfP4bKkhtnA0bP42lR3rSh"
token_generator_url: "https://yj0vsrf8l4.execute-api.us-east-1.amazonaws.com/Initial/GetTokens"
token_generator_api_key: "dBB5ucuV5L6GyKFKmlrfP4bKkhtnA0bP42lR3rSh"


In [ ]:
COURSE_NAME = 'cis5190_fall2026'
HOMEWORK_NUMBER = 4   # -> homework ID cis5190_fall2026_HW4

grader = PennGrader('notebook-config.yaml', HOMEWORK_NUMBER, STUDENT_ID, STUDENT_ID, COURSE_NAME)

# **1. Natural Language Processing [22 pts]**

#### The Stanford Sentiment Treebank (SST)

The [Stanford Sentiment Treebank](https://nlp.stanford.edu/sentiment/index.html) (Socher et al.,
2013) contains about 11,800 sentences from movie reviews. It is a *treebank*: every sentence is
parsed into a tree, and every **phrase** in the tree — not just the whole sentence — carries a
sentiment label. We use the **binary** version: each example is labeled **0 (negative)** or
**1 (positive)**, and we train and evaluate on all phrases, not only whole sentences.

For example, the sentence `it 's a lovely film with lovely performances .` is one example, and its
sub-phrase `'s a lovely film with lovely performances .` is another.

The data is already tokenized and lower-cased. Each example is stored as its list of `tokens` and
as `ids`, the position of each token in a vocabulary of the 20,000 most frequent training words.

#### Download the data

One zip holds everything Part 1 reads: the SST train / dev / test splits, the training tokens the
vocabulary is built from, and the part of the GloVe embedding file that covers this vocabulary
(see 1.1.1). The test split has no labels; the autograder holds them.

In [ ]:
import zipfile

DATA_ZIP = "cis5190_hw4_data.zip"
DATA_URL = "https://raw.githubusercontent.com/CIS5190/cis-4190-5190-fall-26/main/hw4/cis5190_hw4_data.zip"

if NOTEBOOK:
    if not os.path.exists(DATA_ZIP):
        !wget -q {DATA_URL}
    if not os.path.exists("sst_train.parquet"):
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(".")
    print(sorted(f for f in os.listdir(".") if f.startswith(("sst_", "glove", "train_tokens"))))

Below is some helper code to build the vocabulary and the dataset. You do not need to modify it.

In [ ]:
PAD_ID = 0   # the padding id; it is also the id of the start token "<s>", which never
             # appears inside an example, so padding cannot be confused with a real token


class Vocabulary:
    """Maps words to integer ids, most frequent first, after three special tokens."""

    SPECIALS = ["<s>", "</s>", "<unk>"]

    def __init__(self, tokens, size):
        counts = collections.Counter(tokens)
        words = self.SPECIALS + [w for w, _ in counts.most_common(size - len(self.SPECIALS))]
        self.id_to_word = dict(enumerate(words))
        self.word_to_id = {w: i for i, w in self.id_to_word.items()}
        self.size = len(words)

    def ordered_words(self):
        """The list of words, ordered by id: ordered_words()[i] is the word with id i."""
        return [self.id_to_word[i] for i in range(self.size)]


def pad_ids(id_lists, max_len=40):
    """(n, max_len) array of ids, right-padded with PAD_ID, and the true lengths."""
    x = np.full((len(id_lists), max_len), PAD_ID, dtype=np.int64)
    lengths = np.zeros(len(id_lists), dtype=np.int64)
    for i, ids in enumerate(id_lists):
        n = min(len(ids), max_len)
        x[i, :n] = ids[:n]
        lengths[i] = n
    return x, lengths


class SSTData:
    """The three splits as DataFrames, and the vocabulary built from the training tokens."""

    def __init__(self, vocab_size=20000):
        self.train = pd.read_parquet("sst_train.parquet")
        self.dev = pd.read_parquet("sst_dev.parquet")
        self.test = pd.read_parquet("sst_test.parquet")
        with open("train_tokens.txt", encoding="utf-8") as f:
            self.vocab = Vocabulary([w.strip() for w in f], size=vocab_size)

    def as_padded_array(self, split, max_len=40):
        df = getattr(self, split)
        x, lengths = pad_ids(list(df["ids"]), max_len)
        y = df["label"].to_numpy(dtype=np.int64) if "label" in df else None
        return x, lengths, y

In [ ]:
if NOTEBOOK:
    sst = SSTData()
    print(f"vocabulary: {sst.vocab.size:,} words")
    for split in ["train", "dev", "test"]:
        df = getattr(sst, split)
        print(f"{split:>5}: {len(df):>6,} phrases, {int(df['is_root'].sum()):>5,} of them whole sentences")
    print(sst.train.head(3)[["label", "tokens"]])

## **1.1. Deep Averaging Networks [11 pts]**

A [Deep Averaging Network](https://people.cs.umass.edu/~miyyer/pubs/2015_acl_dan.pdf) (DAN;
Iyyer et al., 2015) classifies a text in three steps:

1. look up an **embedding** for every token and take their **average**;
2. pass the average through one or more feed-forward layers;
3. classify the result with a final linear layer.

The average ignores word order — a DAN treats a phrase as a bag of word vectors — and yet it is a
strong, fast baseline for sentiment. Its quality depends a great deal on starting from good word
embeddings, so we initialize ours from pre-trained **GloVe** vectors.

### **1.1.1. Read the GloVe Embeddings [2 pts, Autograded]**

[GloVe](https://nlp.stanford.edu/projects/glove/) (Pennington et al., 2014) publishes word vectors
trained on 840 billion tokens of web text. The file is plain text, one word per line: the word,
then its 300 numbers, all separated by single spaces:

```
the 0.27204 -0.06203 -0.1884 ... 0.13455
```

The full file is 2.2 GB, so the zip contains only its lines for words that could be in our
vocabulary — about 46,700 lines, in their original order. GloVe is case-sensitive and our
vocabulary is lower-case, so the subset still contains words such as `The` and `November` that
are **not** in the vocabulary. A few lines also have a "word" that itself contains spaces, so they
do not split into a word and 300 numbers.

Complete `get_glove_mapping`: return a dictionary from each word **in `vocab`** to its vector as a
`numpy` array of floats. Skip words not in `vocab` (keep the exact spelling — do not lower-case
the file's words), and skip any line that does not parse; the provided `try`/`except ValueError`
does that for you if converting the numbers raises.

In [ ]:
def get_glove_mapping(vocab, file):
    """
    INPUT:
    vocab - a set of words
    file  - path of a GloVe-format text file

    OUTPUT:
    glove_map - dict: word -> numpy float array, for every word of the file that is in vocab
    """
    glove_map = {}
    with open(file, "rb") as fi:
        for line in fi:
            try:
                # STUDENT TODO START:
                # 1. Decode the bytes to a string and split it on spaces
                # 2. The first field is the word; skip the line if the word is not in vocab
                # 3. The remaining fields are the vector: convert them to a float array
                # 4. Store the vector in glove_map under the word
                raise NotImplementedError("1.1.1: get_glove_mapping")
                # STUDENT TODO END
            except ValueError:
                # A line whose word contains spaces does not convert to floats; skip it.
                pass
    return glove_map

In [ ]:
# Local test: a 7-line file. Exactly 4 of its lines belong in the mapping.
def test_glove_mapping_local():
    lines = ["the 0.1 0.2 0.3", "The 9.0 9.0 9.0", "cat 0.5 -0.5 1.0", ". . . 1.0 2.0 3.0",
             "dog 1.0 1.0 1.0", ". 0.0 0.125 -0.25", "sat 0.25 0.5 0.75"]
    with open("glove_local_test.txt", "w") as f:
        f.write("\n".join(lines) + "\n")
    got = get_glove_mapping({"the", "cat", "sat", ".", "<unk>"}, "glove_local_test.txt")
    assert sorted(got) == [".", "cat", "sat", "the"], f"wrong words: {sorted(got)}"
    assert np.allclose(np.asarray(got["."], dtype=float), [0.0, 0.125, -0.25]), "wrong vector for '.'"
    assert np.allclose(np.asarray(got["the"], dtype=float), [0.1, 0.2, 0.3]), "wrong vector for 'the'"
    print("get_glove_mapping: local test passed")

test_glove_mapping_local()

In [ ]:
# PennGrader Grading Cell
# A 7-line file in GloVe's format: a capitalised word that is not in the vocabulary,
# a row whose token contains spaces (so it does not parse), and a word not in the
# vocabulary at all. Exactly four of the lines belong in the mapping.
_lines = [
    "the 0.1 0.2 0.3",
    "The 9.0 9.0 9.0",
    "cat 0.5 -0.5 1.0",
    ". . . 1.0 2.0 3.0",
    "dog 1.0 1.0 1.0",
    ". 0.0 0.125 -0.25",
    "sat 0.25 0.5 0.75",
]
with open("glove_fixture.txt", "w") as _f:
    _f.write("\n".join(_lines) + "\n")
_fixture_map = get_glove_mapping({"the", "cat", "sat", ".", "<unk>"}, "glove_fixture.txt")
answer = {str(w): np.asarray(v, dtype=float).tolist() for w, v in _fixture_map.items()}
grader.grade(test_case_id='test_glove_mapping', answer=answer)

In [ ]:
# Now read the real file (about 20 seconds). You should find 15,506 of the vocabulary's words.
if NOTEBOOK:
    glove_map = get_glove_mapping(set(sst.vocab.ordered_words()), "glove.840B.300d.sst-subset.txt")
    print(f"{len(glove_map):,} vocabulary words have a GloVe vector")

### **1.1.2. Build the Embedding Matrix [2 pts, Autograded]**

The embedding layer is a matrix with one row per vocabulary word: row $i$ is the vector for the
word with id $i$. Complete `get_weight_matrix`: for each word of `words` (in order), use its GloVe
vector if `glove_map` has one, and otherwise a random vector drawn with `np.random.normal` (mean 0,
standard deviation 1). Return a `(len(words), d_embed)` array.

In [ ]:
def get_weight_matrix(words, d_embed, glove_map):
    """
    INPUT:
    words     - list of words; words[i] is the word with id i
    d_embed   - the length of each embedding vector
    glove_map - dict: word -> vector of length d_embed

    OUTPUT:
    weights_matrix - (len(words), d_embed) array; row i is the embedding of words[i]
    """
    # STUDENT TODO START:
    raise NotImplementedError("1.1.2: get_weight_matrix")
    # STUDENT TODO END
    return weights_matrix

In [ ]:
# Local test: rows with a GloVe vector are copied exactly; the others are random, not zero.
def test_weight_matrix_local():
    glove = {"a": np.arange(8.0), "c": -np.arange(8.0)}
    W = np.asarray(get_weight_matrix(["a", "b", "c"], 8, glove), dtype=float)
    assert W.shape == (3, 8), f"expected shape (3, 8), got {W.shape}"
    assert np.allclose(W[0], np.arange(8.0)) and np.allclose(W[2], -np.arange(8.0)), "GloVe rows not copied"
    assert np.abs(W[1]).sum() > 0, "the row for a word with no GloVe vector should be random, not zeros"
    print("get_weight_matrix: local test passed")

test_weight_matrix_local()

In [ ]:
# PennGrader Grading Cell
_rng = np.random.RandomState(4190)
_words = ["<s>", "the", "cat", "<unk>", "sat"]
_glove = {"the": _rng.normal(size=50), "sat": _rng.normal(size=50), "dog": _rng.normal(size=50)}
np.random.seed(0)
_W = np.asarray(get_weight_matrix(_words, 50, _glove), dtype=float)
answer = {"shape": list(_W.shape), "rows": _W.tolist()}
grader.grade(test_case_id='test_weight_matrix', answer=answer)

### **1.1.3. Create the Embedding Layer [1 pt, Autograded]**

Complete `create_emb_layer`: return an `nn.Embedding` with one row per row of `weights_matrix`,
initialized to `weights_matrix`. Convert the embedding weights to `torch.float32` before
initializing the embedding layer. If `non_trainable` is `True`, the weights must not be updated
during training (set the weight's `requires_grad` to `False`); otherwise they must be.

In [ ]:
def create_emb_layer(weights_matrix, non_trainable=False):
    """
    INPUT:
    weights_matrix - (n_embed, d_embed) numpy array
    non_trainable  - if True, the embedding weights are frozen

    OUTPUT:
    emb_layer - nn.Embedding(n_embed, d_embed) whose weight equals weights_matrix
    """
    # STUDENT TODO START:
    raise NotImplementedError("1.1.3: create_emb_layer")
    # STUDENT TODO END
    return emb_layer

In [ ]:
# Local test
def test_embedding_layer_local():
    W = np.arange(6, dtype=np.float64).reshape(3, 2)
    layer = create_emb_layer(W)
    assert layer.weight.dtype == torch.float32, "embedding weights must be torch.float32"
    assert isinstance(layer, nn.Embedding), "create_emb_layer should return an nn.Embedding"
    assert np.allclose(layer.weight.detach().cpu().numpy(), W), "weights not copied from weights_matrix"
    assert layer.weight.requires_grad, "non_trainable=False: the weights should be trainable"
    assert not create_emb_layer(W, non_trainable=True).weight.requires_grad, "non_trainable=True: freeze the weights"
    print("create_emb_layer: local test passed")

test_embedding_layer_local()

In [ ]:
# PennGrader Grading Cell
# float64, as get_weight_matrix may well return: the layer must still come out float32,
# or the DAN's float32 linear layers fail later with a dtype mismatch.
_W = np.arange(12, dtype=np.float64).reshape(4, 3) / 10
_trainable = create_emb_layer(_W, non_trainable=False)
_frozen = create_emb_layer(_W, non_trainable=True)
answer = {
    "type": type(_trainable).__name__,
    "dtype": str(_trainable.weight.dtype),
    "shape": list(_trainable.weight.shape),
    "weight": _trainable.weight.detach().cpu().double().numpy().tolist(),
    "trainable_requires_grad": bool(_trainable.weight.requires_grad),
    "frozen_requires_grad": bool(_frozen.weight.requires_grad),
    "frozen_weight": _frozen.weight.detach().cpu().double().numpy().tolist(),
}
grader.grade(test_case_id='test_embedding_layer', answer=answer)

### **1.1.4. Masked Averaging [2 pts, Autograded]**

The model receives a batch of examples as a `(batch, max_len)` array of token ids, padded on the
right with `PAD_ID` up to `max_len = 40`. After the embedding lookup we have a
`(batch, max_len, d_embed)` tensor `v` — which includes vectors for the padding positions. Those
must not count towards the average.

So with every batch comes a **mask**: a `(batch, max_len)` tensor of 0s and 1s, where 1 means "a
real token, include it" and 0 means "padding (or a word dropped by word dropout, 1.1.5), leave it
out". The masked average of example $b$ is

$$\bar v_b = \frac{\sum_t m_{b,t}\, v_{b,t}}{\sum_t m_{b,t}} .$$

Complete `masked_mean`:

1. reshape the mask to `(batch, max_len, 1)` and expand it to `(batch, max_len, d_embed)`
   ([`expand`](https://pytorch.org/docs/stable/generated/torch.Tensor.expand.html) helps); convert it
   to `v`'s dtype, since the DataLoader delivers it as integers;
2. the **numerator**: multiply `v` by the expanded mask elementwise and sum over the token
   dimension, giving `(batch, d_embed)`;
3. the **denominator**: the number of tokens kept in each example;
4. return numerator / denominator, shape `(batch, d_embed)`.

Every example keeps at least one token, so the denominator is never zero.

In [ ]:
def masked_mean(v, mask):
    """
    INPUT:
    v    - (batch, max_len, d_embed) float tensor of token embeddings
    mask - (batch, max_len) tensor of 0/1; 1 = include this token in the average

    OUTPUT:
    x - (batch, d_embed) tensor; x[b] is the mean of the v[b, t] with mask[b, t] == 1
    """
    batch, max_len, d_embed = v.shape
    # STUDENT TODO START:
    raise NotImplementedError("1.1.4: masked_mean")
    # STUDENT TODO END
    return x

In [ ]:
# Local test: masked positions are excluded from both the sum and the count.
def test_masked_mean_local():
    v = torch.tensor([[[1., 2.], [3., 4.], [100., 100.]],
                      [[5., 6.], [-1., 0.], [7., 8.]]])
    mask = torch.tensor([[1, 1, 0],
                         [1, 0, 1]])
    got = masked_mean(v, mask)
    assert tuple(got.shape) == (2, 2), f"expected shape (2, 2), got {tuple(got.shape)}"
    assert torch.allclose(got.float(), torch.tensor([[2., 3.], [6., 7.]])), f"wrong averages:\n{got}"
    print("masked_mean: local test passed")

test_masked_mean_local()

In [ ]:
# PennGrader Grading Cell
_v = torch.tensor(np.random.RandomState(5).normal(size=(3, 4, 5)), dtype=torch.float32)
# A LongTensor, as the DataLoader delivers it. 1 = a real token, 0 = padding or dropped.
_mask = torch.tensor([[1, 1, 0, 0],
                      [1, 0, 1, 0],
                      [1, 1, 1, 1]])
answer = masked_mean(_v, _mask).detach().cpu().double().numpy().tolist()
grader.grade(test_case_id='test_masked_mean', answer=answer)

### **1.1.5. The Model and the Data Loader [ungraded]**

This code is provided. `SSTTorchDataset` serves `(ids, mask, label)` triples. On the training
split it also applies **word dropout**: each real token is dropped from the average (its mask set
to 0) with probability `word_dropout`, keeping at least one — a regularizer from the DAN paper.

`DAN` embeds the ids with your `create_emb_layer`, averages them with your `masked_mean`, and
passes the average through 1 hidden layer(s) and a final linear layer to 2 class scores.

In [ ]:
class SSTTorchDataset(Dataset):
    def __init__(self, sst, split, word_dropout=0.0):
        self.x, self.lengths, self.y = sst.as_padded_array(split)
        self.word_dropout = word_dropout if split == "train" else 0.0

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        n = self.lengths[i]
        mask = np.zeros(self.x.shape[1], dtype=np.int64)
        mask[:n] = 1
        if self.word_dropout > 0:
            dropped = np.random.rand(n) < self.word_dropout
            if dropped.all():
                dropped[0] = False      # keep at least one token
            mask[:n][dropped] = 0
        label = self.y[i] if self.y is not None else -1   # -1: the test split has no labels
        return self.x[i], mask, label


class DAN(nn.Module):
    def __init__(self, weights_matrix, d_hidden=300, depth=1, dropout=0.2, d_out=2):
        super().__init__()
        self.embed = create_emb_layer(weights_matrix, non_trainable=False)
        layers, d = [], weights_matrix.shape[1]
        for _ in range(depth):
            layers += [nn.Linear(d, d_hidden), nn.ReLU(), nn.Dropout(dropout)]
            d = d_hidden
        self.hidden = nn.Sequential(*layers)
        self.fc_out = nn.Linear(d, d_out)

    def forward(self, ids, mask):
        v = self.embed(ids)                 # (batch, max_len, d_embed)
        avg = masked_mean(v, mask)          # (batch, d_embed)
        return self.fc_out(self.hidden(avg))

### **1.1.6. Train the DAN [ungraded]**

The cell below builds the embedding matrix from your 1.1.2 and trains for 3 epochs with AdamW,
checking dev accuracy every 200 batches and keeping the best model. It takes about 3 minutes on a
CPU. Expect a dev accuracy of roughly 85–87%.

In [ ]:
def evaluate(model, loader):
    """Accuracy on a labeled loader, or the predictions for an unlabeled one."""
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for ids, mask, y in loader:
            preds.append(model(ids.to(device), mask.to(device)).argmax(dim=1).cpu())
            labels.append(y)
    preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    if (labels < 0).all():
        return preds
    return float((preds == labels).mean())


def train_dan(weights_matrix, epochs=3, lr=1e-3, batch_size=64, weight_decay=1e-5,
              word_dropout=0.01, eval_every=200):
    train_loader = DataLoader(SSTTorchDataset(sst, "train", word_dropout), batch_size, shuffle=True)
    dev_loader = DataLoader(SSTTorchDataset(sst, "dev"), 256)
    model = DAN(weights_matrix).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    best_acc, best_state, step = -1.0, None, 0
    for epoch in range(epochs):
        for ids, mask, y in train_loader:
            model.train()
            optimizer.zero_grad()
            loss = criterion(model(ids.to(device), mask.to(device)), y.to(device))
            loss.backward()
            optimizer.step()
            step += 1
            if step % eval_every == 0:
                acc = evaluate(model, dev_loader)
                if acc > best_acc:
                    best_acc = acc
                    best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        print(f"epoch {epoch + 1}: best dev accuracy so far {best_acc:.4f}")
    model.load_state_dict(best_state)
    return model, best_acc

In [ ]:
if NOTEBOOK:
    np.random.seed(42)
    torch.manual_seed(1234)
    weights_matrix = get_weight_matrix(sst.vocab.ordered_words(), 300, glove_map)
    print("embedding matrix:", weights_matrix.shape)
    dan_model, dev_acc = train_dan(weights_matrix)
    print(f"best dev accuracy: {dev_acc:.4f}")

### **1.1.7. Predict the Test Set [4 pts, Autograded]**

The cell below predicts all 26,052 test phrases with your trained model. **Scoring:** the full
**4 points** for a test accuracy of **85% or more**, 0 for
75% or less, linearly in between, rounded to the nearest point. Your dev accuracy is a
good estimate of your test accuracy. This section checks that the whole pipeline trains; the
pieces you wrote are graded on their own in 1.1.1–1.1.4.

**Run this grading cell in Colab** — Gradescope skips it.

In [ ]:
if NOTEBOOK:
    test_loader = DataLoader(SSTTorchDataset(sst, "test"), 256)
    test_predictions = evaluate(dan_model, test_loader)
    assert test_predictions.shape == (len(sst.test),), f"expected {len(sst.test)} predictions"
    print("predicted class counts:", np.bincount(test_predictions, minlength=2))

In [ ]:
# PennGrader Grading Cell -- run this in Colab; Gradescope skips it.
if NOTEBOOK:
    grader.grade(test_case_id='test_dan_predictions', answer=[int(p) for p in test_predictions])

## **1.2. Attention and the Transformer [11 pts]**

The Transformer ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)) is built from
**attention** layers. Given queries $Q$, keys $K$ and values $V$ — one row per token — scaled
dot-product attention computes

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V ,$$

where $d_k$ is the length of each query and key vector, and the softmax is taken over the keys
(each query's weights sum to 1). Padding positions are **masked**: they get no attention weight.

**Multi-head attention** projects the input to $Q$, $K$ and $V$ with three learned weight
matrices, splits each into `n_heads` heads of size `head_dim = embed_dim // n_heads`, runs
attention in every head in parallel, concatenates the heads, and applies an output projection.

![transformer architecture](https://d2l.ai/_images/transformer.svg)

The helpers below are provided. Note the shapes: `mha_transform_input` turns a
`(n_batch, n_tokens, embed_dim)` tensor into `(n_batch * n_heads, n_tokens, head_dim)` — one
"batch row" per (example, head) pair, example-major — and `mha_transform_output` undoes it.

In [ ]:
def masked_softmax(x, mask):
    """
    x:    (N, n_queries, n_keys) attention scores, one row per query
    mask: (N, n_keys), 1 = a real key, 0 = padding
    Softmax over the last (key) dimension, with padded keys given zero weight.
    """
    mask = mask.to(dtype=torch.bool)
    x = x.masked_fill(~mask.unsqueeze(1), torch.finfo(x.dtype).min)
    return F.softmax(x, dim=-1)


def mha_transform_input(x, n_heads, head_dim):
    """(n_batch, n_tokens, n_heads * head_dim) -> (n_batch * n_heads, n_tokens, head_dim)"""
    n_batch, n_tokens, _ = x.shape
    x = x.reshape((n_batch, n_tokens, n_heads, head_dim))
    x = x.permute(0, 2, 1, 3)
    return x.reshape((n_batch * n_heads, n_tokens, head_dim))


def mha_transform_output(x, n_heads, head_dim):
    """(n_batch * n_heads, n_tokens, head_dim) -> (n_batch, n_tokens, n_heads * head_dim)"""
    n_concat, n_tokens, _ = x.shape
    n_batch = n_concat // n_heads
    x = x.reshape((n_batch, n_heads, n_tokens, head_dim))
    x = x.permute(0, 2, 1, 3)
    return x.reshape((n_batch, n_tokens, n_heads * head_dim))

### **1.2.1. Scaled Dot-Product Attention [3 pts, Autograded]**

Source/key and target/query lengths may differ: `n_queries` need not equal `n_keys`.
Each example has at least one unmasked key.

Complete `ScaledDotProductAttention.forward`, implementing the formula above:

1. the batched product $Q K^\top$, shape `(N, n_queries, n_keys)` (`torch.bmm` helps);
2. divide by $\sqrt{d_k}$, where $d_k$ is `self.head_dim`;
3. `masked_softmax` with the mask, giving the attention weights;
4. the batched product of the weights with $V$.

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, head_dim):
        super().__init__()
        self.head_dim = head_dim

    def forward(self, queries, keys, values, mask):
        """
        INPUT:
        queries              - (N, n_queries, head_dim) tensor
        keys, values         - (N, n_keys, head_dim) tensors
        mask                 - (N, n_keys) tensor of 0/1 over the keys

        OUTPUT:
        attention - (N, n_queries, head_dim) tensor
        """
        # STUDENT TODO START:
        raise NotImplementedError("1.2.1: ScaledDotProductAttention.forward")
        # STUDENT TODO END
        return attention

In [ ]:
# Local test: with one real key, every query attends only to it; scores are scaled by sqrt(head_dim).
def test_scaled_dot_product_local():
    att = ScaledDotProductAttention(head_dim=2)
    q = torch.tensor([[[1., 0.], [0., 1.]]])
    k = torch.tensor([[[2., 0.], [0., 2.]]])
    v = torch.tensor([[[1., 10.], [3., 30.]]])
    out = att(q, k, v, torch.tensor([[1, 0]]))
    assert tuple(out.shape) == (1, 2, 2), f"expected shape (1, 2, 2), got {tuple(out.shape)}"
    assert torch.allclose(out, torch.tensor([[[1., 10.], [1., 10.]]])), "the padded key got some weight"
    out = att(q, k, v, torch.tensor([[1, 1]]))
    w = torch.softmax(torch.tensor([2., 0.]) / math.sqrt(2), dim=0)   # query 0's weights
    assert torch.allclose(out[0, 0], w[0] * v[0, 0] + w[1] * v[0, 1], atol=1e-5), \
        "wrong weights: check the scaling by sqrt(head_dim)"
    print("ScaledDotProductAttention: local test passed")

test_scaled_dot_product_local()

In [ ]:
# PennGrader Grading Cell
_rng = np.random.RandomState(7)
_q, _k, _v = (torch.tensor(_rng.normal(size=(2, 5, 4)), dtype=torch.float32) for _ in range(3))
_mask = torch.tensor([[1, 1, 1, 1, 0],
                      [1, 1, 1, 0, 0]])
answer = (ScaledDotProductAttention(head_dim=4)(_q, _k, _v, _mask)
          .detach().cpu().double().numpy().tolist())
grader.grade(test_case_id='test_scaled_dot_product', answer=answer)

### **1.2.2. Multi-Head Attention [6 pts, Autograded]**

Complete `MultiHeadAttention`. Assume `embed_dim` is divisible by `n_heads`.
Queries may have a different token count from keys and values; the output keeps the query count:

- in `__init__`, define the three projections as `nn.Linear(embed_dim, embed_dim, bias=False)`,
  named **exactly** `self.query_fc`, `self.key_fc` and `self.value_fc`. The grading cell sets
  their weights (and `self.out_fc`'s) to fixed values by these names, so that your answer does not
  depend on the random initialization;
- in `forward`:
  1. project `queries` with `query_fc`, `keys` with `key_fc` and `values` with `value_fc`;
  2. split each into heads with `mha_transform_input`;
  3. the mask is `(n_batch, n_keys)`, but the heads are `(n_batch * n_heads, ...)`: give each
     head its example's mask row with `mask.repeat_interleave(self.n_heads, dim=0)`;
  4. run `self.attention` on the heads with that mask;
  5. merge the heads with `mha_transform_output`, and apply `self.out_fc`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, embed_dim):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.attention = ScaledDotProductAttention(self.head_dim)

        # STUDENT TODO START:
        # self.query_fc, self.key_fc, self.value_fc: nn.Linear(embed_dim, embed_dim, bias=False)
        raise NotImplementedError("1.2.2: MultiHeadAttention.__init__")
        # STUDENT TODO END

        self.out_fc = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, queries, keys, values, mask):
        """
        INPUT:
        queries              - (n_batch, n_queries, embed_dim) tensor
        keys, values         - (n_batch, n_keys, embed_dim) tensors
        mask                 - (n_batch, n_keys) tensor of 0/1 over the keys

        OUTPUT:
        attention - (n_batch, n_queries, embed_dim) tensor
        """
        # STUDENT TODO START:
        raise NotImplementedError("1.2.2: MultiHeadAttention.forward")
        # STUDENT TODO END
        return attention

In [ ]:
# Local test: with identity projections and one head, MultiHeadAttention is plain attention.
def test_multihead_attention_local():
    mha = MultiHeadAttention(n_heads=1, embed_dim=3)
    with torch.no_grad():
        for name in ["query_fc", "key_fc", "value_fc", "out_fc"]:
            getattr(mha, name).weight.copy_(torch.eye(3))
    x = torch.tensor(np.random.RandomState(0).normal(size=(2, 4, 3)), dtype=torch.float32)
    mask = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]])
    expected = ScaledDotProductAttention(3)(x, x, x, mask)
    got = mha(x, x, x, mask)
    assert tuple(got.shape) == (2, 4, 3), f"expected shape (2, 4, 3), got {tuple(got.shape)}"
    assert torch.allclose(got, expected, atol=1e-5), "with identity weights this should equal plain attention"
    out2 = MultiHeadAttention(n_heads=2, embed_dim=4)(torch.zeros(2, 4, 4), torch.zeros(2, 4, 4),
                                                      torch.zeros(2, 4, 4), mask)
    assert tuple(out2.shape) == (2, 4, 4), "2 heads, 2 examples: check the mask is repeated per head"
    print("MultiHeadAttention: local test passed")

test_multihead_attention_local()

In [ ]:
# PennGrader Grading Cell
_rng = np.random.RandomState(11)
_mha = MultiHeadAttention(n_heads=2, embed_dim=4)
with torch.no_grad():   # fixed weights, set by name -- see 1.2.2
    for _name in ["query_fc", "key_fc", "value_fc", "out_fc"]:
        getattr(_mha, _name).weight.copy_(torch.tensor(_rng.normal(size=(4, 4)), dtype=torch.float32))
_x = torch.tensor(_rng.normal(size=(2, 5, 4)), dtype=torch.float32)
_y = torch.tensor(_rng.normal(size=(2, 5, 4)), dtype=torch.float32)
_mask = torch.tensor([[1, 1, 1, 1, 0],
                      [1, 1, 0, 0, 0]])
# Queries from _x; keys and values from _y, so mixing up Q, K and V changes the answer.
answer = _mha(_x, _y, _y, _mask).detach().cpu().double().numpy().tolist()
grader.grade(test_case_id='test_multihead_attention', answer=answer)

### **1.2.3. Transformer Integration [2 pts, Autograded]**

The code below assembles a simplified encoder–decoder Transformer integration exercise from
your `MultiHeadAttention`. Decoder causal masking is intentionally omitted: target tokens can
attend to later target tokens, so this is not an autoregressive generation model. It includes
positional encodings, residual connections with layer normalization, and feed-forward layers. You
write nothing here — this section checks that your attention works inside the supplied model, on a
batch of two sequences of different lengths. If 1.2.2 passes and this does not, look at how your
code handles masks, batches and the separate `queries` / `keys` / `values` arguments (the decoder
attends from its own tokens to the encoder's).

In [ ]:
class PositionalEncoder(nn.Module):
    def __init__(self, embed_dim, max_len=300):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1)
        j2 = torch.arange(0, embed_dim, step=2, dtype=torch.float32)
        angle = position / torch.pow(10000, j2 / embed_dim)
        pe = torch.zeros((1, max_len, embed_dim))
        pe[..., 0::2] = torch.sin(angle)
        pe[..., 1::2] = torch.cos(angle)
        self.register_buffer("position_embedding", pe)

    def forward(self, x):
        return x + self.position_embedding[:, : x.shape[1]]


class ResidualNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, residual):
        return self.norm(x + residual)


class Feedforward(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, embed_dim)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class EncoderBlock(nn.Module):
    def __init__(self, n_heads, embed_dim, hidden_dim):
        super().__init__()
        self.attention = MultiHeadAttention(n_heads, embed_dim)
        self.norm1 = ResidualNorm(embed_dim)
        self.feedforward = Feedforward(embed_dim, hidden_dim)
        self.norm2 = ResidualNorm(embed_dim)

    def forward(self, src, src_mask):
        a = self.norm1(self.attention(src, src, src, src_mask), src)   # self-attention
        return self.norm2(self.feedforward(a), a)


class DecoderBlock(nn.Module):
    def __init__(self, n_heads, embed_dim, hidden_dim):
        super().__init__()
        self.self_attention = MultiHeadAttention(n_heads, embed_dim)
        self.norm1 = ResidualNorm(embed_dim)
        self.encoder_attention = MultiHeadAttention(n_heads, embed_dim)
        self.norm2 = ResidualNorm(embed_dim)
        self.feedforward = Feedforward(embed_dim, hidden_dim)
        self.norm3 = ResidualNorm(embed_dim)

    def forward(self, tgt, tgt_mask, encoder_state, src_mask):
        a = self.norm1(self.self_attention(tgt, tgt, tgt, tgt_mask), tgt)
        # queries from the decoder, keys and values from the encoder
        b = self.norm2(self.encoder_attention(a, encoder_state, encoder_state, src_mask), a)
        return self.norm3(self.feedforward(b), b)


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, embed_dim, hidden_dim, n_heads, n_blocks):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, embed_dim)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, embed_dim)
        self.positional_encoding = PositionalEncoder(embed_dim)
        self.encoder_blocks = nn.ModuleList(
            [EncoderBlock(n_heads, embed_dim, hidden_dim) for _ in range(n_blocks)])
        self.decoder_blocks = nn.ModuleList(
            [DecoderBlock(n_heads, embed_dim, hidden_dim) for _ in range(n_blocks)])
        self.out = nn.Linear(embed_dim, tgt_vocab_size)

    def forward(self, src_tokens, src_mask, tgt_tokens, tgt_mask):
        x = self.positional_encoding(self.src_embedding(src_tokens))
        for block in self.encoder_blocks:
            x = block(x, src_mask)
        y = self.positional_encoding(self.tgt_embedding(tgt_tokens))
        for block in self.decoder_blocks:
            y = block(y, tgt_mask, x, src_mask)
        return self.out(y)     # (n_batch, n_tgt_tokens, tgt_vocab_size) scores

In [ ]:
# Local test: the output shape, for a batch of two padded sequences.
def test_transformer_local():
    model = Transformer(src_vocab_size=7, tgt_vocab_size=9, embed_dim=4, hidden_dim=8, n_heads=2, n_blocks=2)
    src = torch.tensor([[1, 2, 3, 0], [4, 5, 0, 0]])
    tgt = torch.tensor([[1, 2, 3, 4, 5], [6, 7, 8, 0, 0]])
    out = model(src, (src != 0).long(), tgt, (tgt != 0).long())
    assert tuple(out.shape) == (2, 5, 9), f"expected shape (2, 5, 9), got {tuple(out.shape)}"
    assert torch.isfinite(out).all(), "the output contains NaN or inf"
    print("Transformer: local test passed")

test_transformer_local()

In [ ]:
# PennGrader Grading Cell
_model = Transformer(src_vocab_size=7, tgt_vocab_size=7, embed_dim=4, hidden_dim=8,
                     n_heads=2, n_blocks=2)
_rng = np.random.RandomState(13)
with torch.no_grad():   # every weight set by name, in sorted order (LayerNorms keep 1 and 0)
    for _name, _p in sorted(_model.named_parameters()):
        if "norm" in _name:
            continue
        _p.copy_(torch.tensor(_rng.normal(scale=0.5, size=tuple(_p.shape)), dtype=torch.float32))
# Source length 4, target length 5, as in the local test: the decoder's cross-attention
# has n_queries != n_keys.
_src = torch.tensor([[1, 2, 3, 4], [5, 6, 1, 0]])
_tgt = torch.tensor([[2, 3, 4, 5, 6], [1, 1, 2, 0, 0]])
_src_mask = (_src != 0).long()
_tgt_mask = (_tgt != 0).long()
answer = _model(_src, _src_mask, _tgt, _tgt_mask).detach().cpu().double().numpy().tolist()
grader.grade(test_case_id='test_transformer', answer=answer)

# **2. Reinforcement Learning [24 pts — 16 for CIS 4190]**

> **Part of this is lectured during the homework window.** Reinforcement learning is covered on
> Thu 5 November (RL Part 1) and Tue 17 November (RL Part 2). Sections 2.1–2.3 need only the first
> lecture; actor-critic (2.5) and the questions in 2.6 draw on the second.

We train an agent on [Mountain Car Continuous](https://gymnasium.farama.org/environments/classic_control/mountain_car_continuous/):
an under-powered car sits in a valley and must reach the flag on the hill to the right. The
**state** is the car's position and velocity; the **action** is a force in $[-1, 1]$. The car
cannot drive straight up the hill — it has to rock back and forth to build momentum. Each step
costs $0.1\,a^2$ (pushing hard is expensive), reaching the flag earns $+100$ and ends the episode,
and otherwise an episode ends after 999 steps.

We learn a **policy** $\pi_\theta(a \mid s)$ directly, with the policy-gradient algorithm
**REINFORCE**.

In [ ]:
if NOTEBOOK:
    !pip -q install "gymnasium[classic-control]" imageio

In [ ]:
import gymnasium as gym
import sklearn.pipeline
import sklearn.preprocessing
from sklearn.kernel_approximation import RBFSampler
from torch.distributions import Normal

seed = 5190   # don't change this line!!
env = gym.make("MountainCarContinuous-v0", render_mode="rgb_array")
print("state space:", env.observation_space, " action space:", env.action_space)

## **2.1. Featurize the State [2 pts, Autograded]**

A linear policy on the raw (position, velocity) pair cannot represent much. So we map each state
to 400 features: first **standardize** it (fitted below on 10,000 states drawn uniformly from the
state space), then apply four sets of 100 random features that approximate RBF kernels of
different widths. Both transforms are fitted for you.

Complete `featurize_state`: apply `scaler`, then `featurizer`, to one state, and return the 400
features as a 1-D array. Both expect a 2-D input of shape `(n_samples, 2)`.

In [ ]:
rng = np.random.RandomState(seed)
observation_examples = rng.uniform(env.observation_space.low, env.observation_space.high, size=(10000, 2))
scaler = sklearn.preprocessing.StandardScaler()
scaler.fit(observation_examples)

featurizer = sklearn.pipeline.FeatureUnion([
    ("rbf1", RBFSampler(gamma=5.0, n_components=100, random_state=seed)),
    ("rbf2", RBFSampler(gamma=2.0, n_components=100, random_state=seed)),
    ("rbf3", RBFSampler(gamma=1.0, n_components=100, random_state=seed)),
    ("rbf4", RBFSampler(gamma=0.5, n_components=100, random_state=seed)),
])
featurizer.fit(scaler.transform(observation_examples))

In [ ]:
def featurize_state(state):
    """
    INPUT:
    state - array of shape (2,): (position, velocity)

    OUTPUT:
    features - array of shape (400,)
    """
    # STUDENT TODO START:
    raise NotImplementedError("2.1: featurize_state")
    # STUDENT TODO END
    return features

In [ ]:
# Local test
def test_featurize_state_local():
    s = np.array([-0.5, 0.0])
    got = np.asarray(featurize_state(s))
    assert got.shape == (400,), f"expected shape (400,), got {got.shape}"
    expected = featurizer.transform(scaler.transform(s.reshape(1, -1)))[0]
    assert np.allclose(got, expected), "standardize first, then featurize"
    print("featurize_state: local test passed")

test_featurize_state_local()

In [ ]:
# PennGrader Grading Cell
answer = np.asarray(featurize_state(np.array([-0.8248636, 0.02986798])), dtype=float).tolist()
grader.grade(test_case_id='test_featurize_state', answer=answer)

## **2.2. A Gaussian Policy [4 pts, Autograded]**

Our policy is a Normal distribution over the action, whose mean $\mu(s)$ and standard deviation
$\sigma(s)$ are each a linear function of the state's features $\phi(s)$:

$$\pi_\theta(a \mid s) = \mathcal{N}\big(a;\ \mu(s), \sigma(s)^2\big), \qquad
\mu(s) = w_\mu^\top \phi(s) + b_\mu, \qquad
\sigma(s) = \text{softplus}\big(w_\sigma^\top \phi(s) + b_\sigma\big) + 10^{-5} .$$

The softplus keeps $\sigma$ positive. REINFORCE increases the log-probability of each action in
proportion to the return $G$ that followed it, so for one step the **loss** to minimize is

$$\ell = -\big(\log \pi_\theta(a \mid s)\, G + \beta\, H[\pi_\theta(\cdot \mid s)]\big),$$

where $H$ is the policy's entropy and $\beta$ is `entropy_rate`: the entropy bonus rewards keeping
some randomness, which helps exploration.

Complete `PolicyEstimator`:

1. **`__init__`:** `self.mu_head` and `self.sigma_head`, each an `nn.Linear(state_dim, 1)`.
   Initialize `mu_head`'s weight and bias, and `sigma_head`'s weight, to **zero**, and
   `sigma_head`'s bias to $\text{softplus}^{-1}(0.8) = \log(e^{0.8} - 1)$ — so the untrained policy
   is $\mathcal{N}(0, 0.8^2)$ in every state (`nn.init.zeros_` and `nn.init.constant_` help).
2. **`forward`:** compute `mu`, `sigma` as above (`F.softplus`), and `dist = Normal(mu, sigma)`.
3. **`update(state, target, action)`:** zero the gradients; featurize the state **with
   `self.featurize_state`** into a `(1, state_dim)` float tensor; run `forward`; compute the loss
   above with $G$ = `target` and $a$ = `action` (`dist.log_prob`, `dist.entropy`); back-propagate;
   take an optimizer step; return the loss as a float.

Two sampling helpers are provided. `sample_action(state)` returns the original, **unclipped**
Gaussian sample. Use that sample in `update` when evaluating `dist.log_prob`.
For the environment only, clip it to $[-1, 1]$ with `np.clip`; do not overwrite the original
sample. `predict(state)` returns a clipped action for evaluation and behavioral cloning.
This keeps the Gaussian log-probability paired with the sample actually drawn from it, while
keeping the force and its action penalty bounded in the environment. The entropy bonus is
also the entropy of the original Gaussian.

In [ ]:
class PolicyEstimator(nn.Module):
    def __init__(self, state_dim=400, learning_rate=1e-3, entropy_rate=0.1):
        super().__init__()
        # STUDENT TODO START:
        raise NotImplementedError("2.2: PolicyEstimator.__init__")
        # STUDENT TODO END

        self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)
        self.featurize_state = featurize_state
        self.entropy_rate = entropy_rate

    def forward(self, state_tensor):
        """
        INPUT:
        state_tensor - (N, state_dim) float tensor of features

        OUTPUT:
        dist  - torch.distributions.Normal(mu, sigma)
        mu    - (N, 1) tensor
        sigma - (N, 1) tensor
        """
        x = state_tensor
        # STUDENT TODO START:
        raise NotImplementedError("2.2: PolicyEstimator.forward")
        # STUDENT TODO END
        return dist, mu, sigma

    @torch.no_grad()
    def sample_action(self, state):
        """Provided. Original Gaussian sample, retained for the policy update."""
        s = torch.as_tensor(self.featurize_state(state), dtype=torch.float32).unsqueeze(0)
        dist, _, _ = self.forward(s)
        return dist.sample().squeeze()

    @torch.no_grad()
    def predict(self, state):
        """Provided. Clipped action for evaluation and behavioral cloning."""
        return self.sample_action(state).clamp(-1.0, 1.0)

    def update(self, state, target, action):
        """
        One gradient step on the REINFORCE loss for a single (state, action, return).

        INPUT:
        state  - raw state, shape (2,)
        target - float, the return G
        action - float, the original Gaussian sample before environment clipping

        OUTPUT:
        loss - float
        """
        # STUDENT TODO START:
        raise NotImplementedError("2.2: PolicyEstimator.update")
        # STUDENT TODO END
        return float(loss.item())

In [ ]:
# Local test: the initial policy, and the direction of one update.
def test_policy_estimator_local():
    torch.manual_seed(0)
    pe = PolicyEstimator(state_dim=3, learning_rate=0.1)
    pe.featurize_state = lambda s: np.asarray(s, dtype=np.float32)
    x = torch.tensor([[0.5, -1.0, 2.0]])
    with torch.no_grad():
        dist, mu, sigma = pe(x)
    assert tuple(mu.shape) == (1, 1) and tuple(sigma.shape) == (1, 1), "mu and sigma should be (N, 1)"
    assert abs(float(mu)) < 1e-6 and abs(float(sigma) - 0.8) < 1e-4, \
        f"the untrained policy should be N(0, 0.8^2), got mu={float(mu):.4f}, sigma={float(sigma):.4f}"
    pe.update(np.array([0.5, -1.0, 2.0]), 5.0, 0.6)      # a good return after action 0.6 ...
    with torch.no_grad():
        _, mu1, _ = pe(x)
    assert float(mu1) > 0, "... should move the mean towards 0.6 -- check the sign of the loss"
    print("PolicyEstimator: local test passed")

test_policy_estimator_local()

In [ ]:
# PennGrader Grading Cell
torch.manual_seed(0)
_pe = PolicyEstimator(state_dim=3, learning_rate=0.1, entropy_rate=0.1)
_pe.featurize_state = lambda s: np.asarray(s, dtype=np.float32)   # 3 raw numbers as features
_x = np.array([0.5, -1.0, 2.0], dtype=np.float32)
_xt = torch.tensor(_x).unsqueeze(0)
with torch.no_grad():
    _, _mu0, _sigma0 = _pe(_xt)
_loss = _pe.update(_x, 2.0, 0.3)       # state, target (the return G), action
with torch.no_grad():
    _, _mu1, _sigma1 = _pe(_xt)
answer = {
    "mu0": float(_mu0.squeeze()), "sigma0": float(_sigma0.squeeze()),
    "loss": float(_loss),
    "mu1": float(_mu1.squeeze()), "sigma1": float(_sigma1.squeeze()),
}
grader.grade(test_case_id='test_policy_estimator', answer=answer)

## **2.3. Discounted Returns [2 pts, Autograded]**

The return from step $t$ of an episode with rewards $r_0, \dots, r_{T-1}$ is

$$G_t = \sum_{k=t}^{T-1} \gamma^{\,k-t} r_k = r_t + \gamma\, G_{t+1}, \qquad G_T = 0 .$$

Complete `compute_returns`: return all $T$ returns as a `numpy` array. The recursion, run
backwards from the last step, does it in one pass.

In [ ]:
def compute_returns(rewards, discount_factor):
    """
    INPUT:
    rewards         - list of T floats, r_0 ... r_{T-1}
    discount_factor - gamma

    OUTPUT:
    returns - (T,) numpy array; returns[t] = G_t
    """
    # STUDENT TODO START:
    raise NotImplementedError("2.3: compute_returns")
    # STUDENT TODO END
    return returns

In [ ]:
# Local test
def test_compute_returns_local():
    got = np.asarray(compute_returns([1.0, 2.0, 3.0], 0.5), dtype=float)
    assert got.shape == (3,), f"expected shape (3,), got {got.shape}"
    assert np.allclose(got, [1 + 0.5 * 2 + 0.25 * 3, 2 + 0.5 * 3, 3]), f"wrong returns: {got}"
    print("compute_returns: local test passed")

test_compute_returns_local()

In [ ]:
# PennGrader Grading Cell
answer = {
    "gamma_0.9": np.asarray(compute_returns([1.0, 0.0, -2.0, 3.0, 0.5], 0.9), dtype=float).tolist(),
    "gamma_1.0": np.asarray(compute_returns([-1.0, -1.0, -1.0, 100.0], 1.0), dtype=float).tolist(),
}
grader.grade(test_case_id='test_compute_returns', answer=answer)

## **2.4. REINFORCE [8 pts: 4 pts Autograded, 4 pts Manually Graded]**

Complete `reinforce`. For each episode:

1. **Play** the episode with the current policy: from the `state` the provided reset gives you,
   repeatedly draw `action = float(estimator_policy.sample_action(state))`, step the environment
   with `env.step([float(np.clip(action, -1.0, 1.0))])`, and record the state, original
   **unclipped** action and reward, until the step reports
   `terminated` (the flag) or `truncated` (999 steps).
2. Record the episode's **undiscounted return** (the plain sum of its rewards) and its **length**
   in `stats`.
3. Compute the discounted returns $G_t$ with your `compute_returns`, then **normalize** them:
   subtract their mean and divide by their standard deviation (`np.std`, plus `1e-8`).
   We use this within-episode standardization as a practical training heuristic, not a
   guarantee of variance reduction or an independently estimated state-value baseline.
4. **Update** the policy once per step: `estimator_policy.update(state_t, G_t, action_t)`, using the normalized return and
   the original, unclipped Gaussian sample.

In [ ]:
def reinforce(env, estimator_policy, num_episodes, discount_factor):
    """
    OUTPUT:
    stats - dict with "episode_rewards" (undiscounted return of each episode) and
            "episode_lengths" (number of steps in each episode), each of length num_episodes
    """
    stats = {"episode_rewards": np.zeros(num_episodes), "episode_lengths": np.zeros(num_episodes)}
    for i_episode in range(num_episodes):
        state, _ = env.reset(seed=seed + i_episode)
        # STUDENT TODO START:
        raise NotImplementedError("2.4: reinforce")
        # STUDENT TODO END
        print(f"\rEpisode {i_episode + 1}/{num_episodes}: return {stats['episode_rewards'][i_episode]:8.2f}, "
              f"{int(stats['episode_lengths'][i_episode])} steps", end="")
    print()
    return stats

In [ ]:
# Local test: a scripted 2-step episode. The environment gets the clipped action; update gets
# the original sample and the normalized return.
def test_reinforce_local():
    class Env:
        def reset(self, seed=None):
            self.t, self.actions = 0, []
            return np.zeros(2), {}
        def step(self, action):
            self.actions.append(float(np.ravel(action)[0]))
            self.t += 1
            return np.zeros(2), [1.0, 3.0][self.t - 1], self.t == 2, False, {}

    class Policy:
        def __init__(self):
            self.samples, self.updates = iter([2.0, -0.5]), []
        def sample_action(self, state):
            return torch.tensor(next(self.samples))
        def predict(self, state):
            return self.sample_action(state).clamp(-1.0, 1.0)
        def update(self, state, target, action):
            self.updates.append((float(target), float(action)))
            return 0.0

    env, policy = Env(), Policy()
    stats = reinforce(env, policy, num_episodes=1, discount_factor=0.5)
    assert list(stats["episode_rewards"]) == [4.0] and list(stats["episode_lengths"]) == [2], \
        f"stats should hold the undiscounted return 4 and the length 2, got {stats}"
    assert env.actions == [1.0, -0.5], f"the environment should get the CLIPPED action, got {env.actions}"
    assert sorted(a for _, a in policy.updates) == [-0.5, 2.0], \
        "update should get the ORIGINAL sample from sample_action, not the clipped action"
    assert np.allclose(sorted(g for g, _ in policy.updates), [-1.0, 1.0]), \
        "the returns (2.5, 3.0) should be normalized to mean 0 and standard deviation 1"
    print("reinforce: local test passed")

test_reinforce_local()

In [ ]:
# PennGrader Grading Cell
class _ScriptedEnv:
    """Two episodes whose rewards are fixed in advance, whatever the actions: the first
    reaches the flag (terminated) after 3 steps, the second runs out of time (truncated)
    after 2. A state's first entry encodes (episode, step), so updates can be matched."""
    def __init__(self):
        self.script = [[1.0, -2.0, 3.0], [0.5, 0.5]]
        self.episode, self.t, self.env_actions = -1, 0, []
    def reset(self, seed=None):
        self.episode += 1
        self.t = 0
        return np.array([10.0 * self.episode, 0.0]), {}
    def step(self, action):
        self.env_actions.append(float(np.asarray(action, dtype=float).reshape(-1)[0]))
        rewards = self.script[self.episode]
        if self.t >= len(rewards):
            raise RuntimeError("reinforce kept stepping after the episode ended. Stop the episode "
                               "when env.step reports terminated OR truncated.")
        reward = rewards[self.t]
        self.t += 1
        done = self.t == len(rewards)
        state = np.array([10.0 * self.episode + self.t, 0.0])
        return state, reward, done and self.episode == 0, done and self.episode == 1, {}

class _RecordingPolicy:
    """Samples a fixed sequence -- two of them outside [-1, 1] -- and records every update."""
    def __init__(self):
        self.samples = iter([2.5, -0.3, 0.7, -1.8, 0.1])
        self.updates = []
    def sample_action(self, state):
        try:
            return torch.tensor(next(self.samples))
        except StopIteration:
            raise RuntimeError("reinforce asked for more actions than the two scripted episodes "
                               "have steps. Stop each episode when env.step reports terminated "
                               "OR truncated, and draw one action per step.") from None
    def predict(self, state):
        return self.sample_action(state).clamp(-1.0, 1.0)
    def update(self, state, target, action):
        self.updates.append([float(np.asarray(state, dtype=float).reshape(-1)[0]),
                             float(target), float(action)])
        return 0.0

_env, _policy = _ScriptedEnv(), _RecordingPolicy()
_stats = reinforce(_env, _policy, num_episodes=2, discount_factor=0.9)
answer = {
    "episode_rewards": np.asarray(_stats["episode_rewards"], dtype=float).tolist(),
    "episode_lengths": np.asarray(_stats["episode_lengths"], dtype=float).tolist(),
    "env_actions": _env.env_actions,
    "updates": sorted(_policy.updates),
}
grader.grade(test_case_id='test_reinforce', answer=answer)

The cell below trains for 50 episodes (about 3 minutes) and plots the result.

**What is graded.** Your `reinforce` is autograded (4 pts) by the cells
above, on a scripted environment. By hand (4 pts): the three plots below, produced by
your `reinforce` (1 pt), and your answer to the question after them (3 pts). **You are not
graded on how high the return gets.** Mountain Car is a hard exploration problem: the agent only sees the $+100$
if random rocking happens to carry it to the flag, and until then every push only costs reward.
With a correct implementation, some runs reach the flag within 50 episodes and many do
not.

In [ ]:
# Plot for 2.4 -- keep the figures in this cell.
if NOTEBOOK:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    policy_estimator = PolicyEstimator(learning_rate=1e-3)
    stats = reinforce(env, policy_estimator, num_episodes=50, discount_factor=0.99)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(stats["episode_rewards"])
    axes[0].set_title("Undiscounted return per episode")
    axes[1].plot(pd.Series(stats["episode_rewards"]).rolling(5, min_periods=1).mean())
    axes[1].set_title("Return, rolling mean over 5 episodes")
    axes[2].plot(stats["episode_lengths"])
    axes[2].axhline(999, color="gray", linestyle="--", label="999 = never reached the flag")
    axes[2].set_title("Episode length")
    axes[2].legend()
    for ax in axes:
        ax.set_xlabel("episode")
    plt.show()

**Question (2.4).** Describe what your three plots show. Did your agent ever reach the flag, and
if so, did it keep reaching it? Using the reward structure described at the start of Part 2,
explain why a policy that never reaches the flag ends up with a return near $-100$ or near $0$
depending on how hard it pushes, and why that makes this problem hard for REINFORCE.

**Your answer (2.4):** *replace this italic line with your answer.*

## **2.5. Behavioral Cloning and Actor-Critic [ungraded]**

No code to write here — run these cells; section 2.6 asks about the results.

**Behavioral cloning (BC)** is supervised learning on an expert's behavior: we roll out a policy
(the "expert"), record its (state, action) pairs, and fit a regression model — here a decision
tree — that predicts the action from the state. We clone your REINFORCE policy, then train an
**actor-critic** agent, which also learns a value function $V(s)$ and updates the policy with the
TD error $r + \gamma V(s') - V(s)$ after every step, and clone that too.

In [ ]:
def collect_expert_data(env, policy, n_episodes=10):
    """Roll out a policy; return its (states, actions) and each episode's return."""
    states, actions, returns = [], [], []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed + 1000 + i)
        total = 0.0
        while True:
            a = float(np.ravel(policy.predict(obs))[0])
            states.append(np.asarray(obs, dtype=np.float32))
            actions.append(a)
            obs, r, terminated, truncated, _ = env.step([a])
            total += float(r)
            if terminated or truncated:
                break
        returns.append(total)
    return np.array(states), np.array(actions), returns


class TreePolicy:
    """A fitted regressor used as a policy: predict(state) -> action in [-1, 1]."""
    def __init__(self, model):
        self.model = model

    def predict(self, state):
        return np.clip(self.model.predict(np.asarray(state).reshape(1, -1)), -1.0, 1.0)


def evaluate_policy(env, policy, n_episodes=10):
    _, _, returns = collect_expert_data(env, policy, n_episodes)
    return float(np.mean(returns))

In [ ]:
import sklearn.tree

if NOTEBOOK:
    np.random.seed(seed)
    torch.manual_seed(seed)
    states, actions, expert_returns = collect_expert_data(env, policy_estimator)
    bc_reinforce = TreePolicy(sklearn.tree.DecisionTreeRegressor(random_state=seed).fit(states, actions))
    results = {"REINFORCE": float(np.mean(expert_returns)),
               "BC from REINFORCE": evaluate_policy(env, bc_reinforce)}
    print(results)

In [ ]:
class ValueEstimator(nn.Module):
    """V(s) = w^T phi(s) + b, trained by gradient steps on (V(s) - target)^2."""
    def __init__(self, state_dim=400, learning_rate=1e-1):
        super().__init__()
        self.value_head = nn.Linear(state_dim, 1)
        nn.init.zeros_(self.value_head.weight)
        nn.init.zeros_(self.value_head.bias)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)

    def _features(self, state):
        return torch.as_tensor(featurize_state(state), dtype=torch.float32).unsqueeze(0)

    @torch.no_grad()
    def predict(self, state):
        return float(self.value_head(self._features(state)))

    def update(self, state, target):
        self.optimizer.zero_grad()
        loss = F.mse_loss(self.value_head(self._features(state)), torch.tensor([[float(target)]]))
        loss.backward()
        self.optimizer.step()


def actor_critic(env, estimator_policy, estimator_value, num_episodes, discount_factor):
    stats = {"episode_rewards": np.zeros(num_episodes), "episode_lengths": np.zeros(num_episodes)}
    for i_episode in range(num_episodes):
        state, _ = env.reset(seed=seed + i_episode)
        while True:
            action = float(estimator_policy.sample_action(state))
            next_state, reward, terminated, truncated, _ = env.step([float(np.clip(action, -1.0, 1.0))])
            stats["episode_rewards"][i_episode] += reward
            stats["episode_lengths"][i_episode] += 1
            v_next = 0.0 if terminated else estimator_value.predict(next_state)
            td_target = reward + discount_factor * v_next
            td_error = td_target - estimator_value.predict(state)
            estimator_value.update(state, td_target)                 # the critic
            estimator_policy.update(state, td_error, action)         # the actor
            if terminated or truncated:
                break
            state = next_state
        print(f"\rEpisode {i_episode + 1}/{num_episodes}: return {stats['episode_rewards'][i_episode]:8.2f}", end="")
    print()
    return stats

In [ ]:
if NOTEBOOK:
    np.random.seed(seed)
    torch.manual_seed(seed)
    ac_policy = PolicyEstimator(learning_rate=1e-3)
    ac_stats = actor_critic(env, ac_policy, ValueEstimator(), num_episodes=50, discount_factor=0.99)
    states, actions, ac_returns = collect_expert_data(env, ac_policy)
    bc_ac = TreePolicy(sklearn.tree.DecisionTreeRegressor(random_state=seed).fit(states, actions))
    results["Actor-critic"] = float(np.mean(ac_returns))
    results["BC from actor-critic"] = evaluate_policy(env, bc_ac)
    for name, value in results.items():
        print(f"{name:>22}: mean return over 10 episodes {value:8.2f}")

    # The two training runs side by side, for 2.6.1.
    plt.figure(figsize=(8, 4))
    plt.plot(stats["episode_rewards"], label="REINFORCE (2.4)")
    plt.plot(ac_stats["episode_rewards"], label="actor-critic")
    plt.xlabel("episode")
    plt.ylabel("undiscounted return")
    plt.title("Training return per episode")
    plt.legend()
    plt.show()

## **2.6. Analysis [8 pts, Manually Graded, 5190 Extra]**

#### **2.6.1. Actor-critic [2 pts]**
How does actor-critic's update differ from REINFORCE's, and why can learning a value function
$V(s)$ make policy learning need fewer episodes? Compare your two training runs.

**Your answer (2.6.1):** *replace this italic line with your answer.*

#### **2.6.2. Reward alignment [2 pts]**
Read the [reward definition](https://gymnasium.farama.org/environments/classic_control/mountain_car_continuous/#rewards).
Does the per-step reward encourage the behavior we want? Justify your answer.

**Your answer (2.6.2):** *replace this italic line with your answer.*

#### **2.6.3. Hyperparameters [2 pts]**
Propose two hyperparameter changes to the `reinforce(...)` call or `PolicyEstimator(...)`.
For each, state the direction of the change, explain why it might improve exploration or learning,
and describe a possible tradeoff. You do not need to rerun training.

**Your answer (2.6.3):** *replace this italic line with your answer.*

#### **2.6.4. Behavioral cloning [2 pts]**
Report the four mean returns printed in 2.5. Is it possible for a BC policy to out-perform its
expert? Explain why or why not.

**Your answer (2.6.4):** *replace this italic line with your answer.*

# **Before You Submit: Two Short Reports [4 pts, Autograded]**

These two cells are worth **2 points each**, and the points are for *answering*,
not for any particular answer. There is no answer that scores better than
another: reporting that you used no AI earns full marks, and so does reporting
that you used a lot of it.

We use the responses to calibrate how long assignments actually take and to
understand how students are using AI tools.

### **1. Use of AI tools [2 pts]**

Report any use of AI tools (ChatGPT, Claude, Copilot, Gemini, Cursor, etc.) while
working on this assignment: which ones, how you used them, and whether they helped
or got in the way of understanding the material.

**Being honest here will not cost you points.** Writing `"none"` earns full marks,
and so does describing heavy use. We are trying to learn what actually helps people
learn, which only works if the reports are accurate.

In [ ]:
# Replace the empty string with your answer. Anything you write earns full marks --
# including "none". A few sentences is plenty.
AI_USE = ""


# PennGrader grading command
grader.grade('test_ai_use_report', AI_USE)

### **2. Time spent [2 pts]**

Roughly how long did this assignment take you, and how was that time split across
the parts? Rough numbers are genuinely useful — please do not agonize over precision.

You will get a far better estimate if you jot this down **as you go** rather than
reconstructing it at the end.

In [ ]:
# Replace the empty string with your answer. Anything you write earns full marks.
# For example: "about 7 hours -- 3 on Part 1.1, 2 on attention, 2 on RL".
TIME_SPENT_HOURS = ""


# PennGrader grading command
grader.grade('test_time_report', TIME_SPENT_HOURS)

# **Submit to Gradescope**

Congratulations on completing the HW4 coding assignment!

**Before you download — check your figures and answers are there.** Sections 2.4 and 2.6 are
marked by hand from the plots and written answers in this notebook. A notebook downloaded with its
outputs cleared arrives with nothing to mark in those sections, and they are 12 of the
50 points.

**Check that 1.1.7's grading cell ran in Colab** — Gradescope skips it when it re-runs your
notebook.

Download this notebook (`File` → `Download` → `Download .ipynb`) and submit it to
[Gradescope](https://www.gradescope.com).